# Make the timing bar plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import json
from sakura import *
from simplotter.utils.plotttools import legend
setStyle(customized=False)
DRAFT = False #True
THELABEL = "Simulation Preliminary"
add = "_CPU"

In [ ]:
# load data
configDict = {
    "Legacy":"Legacy\n(IT only)\n[2 iterations]", 
    "PatatrackIT":"Patatrack\n(IT only)\n[2 iterations]", 
    "PatatrackIT+3OT": "Patatrack\n(IT + 3 OT layers)\n[1 iteration]"
}
configs = list(configDict.keys())
steps = ["InitialStep", "HighPtTripletStep"]
hlts = ["75e33_timing", "NGTScouting"]  # 
times = {hlt : {} for hlt in hlts}
for config in configs:
    for hlt in hlts:
        with open('timing/%s_%s%s_16j_16t_16s.json' % (config, hlt, add)) as f:
            times[hlt][config] = json.load(f)

with open('timing/Tracking_Groups_StepsSplit.json') as f:
    groupDict = json.load(f)
#with open('timing/Tracking_Colors.json') as f:
with open('timing/tracking_DPNote.json') as f:
    colorDict = json.load(f)

Nevents = {hlt : times[hlt]["Legacy"]["modules"][i]["events"] for hlt, i in zip(hlts, [0,131])}
DIR = "plots/timing%s/" % (add)

In [ ]:
Nevents = {'75e33_timing': 40224, 'NGTScouting': 40224}  # events that the HLT actually ran on (needed to rescale)

In [ ]:
def getGroup(module):
    for g in groupDict.keys():
        if all(s in module["type"] + "|" + module["label"] for s in g.replace('?',' ').replace('*',' ').split()):
            group = groupDict[g]
            if "InitialStep|" in group:
                step = "InitialStep"
                group = group[12:]#
            elif "HighPtTripletStep|" in group:
                step = "HighPtTripletStep"
                group = group[18:]
            else:
                step = "Else"

            if "|" in group:
                group = group[:group.find("|")]
                
            return step, group

    print(module, "not found!")
    return "Else", "Else"

In [ ]:
groups = {hlt : 
          {config : {
              step: {} for step in steps
          } for config in configs}
          for hlt in hlts}
for hlt in hlts:
    for config in configs:
        for module in times[hlt][config]["modules"]:
            step, group = getGroup(module)
            if step!="Else":
                groups[hlt][config][step][group] = module["time_real"] + (groups[hlt][config][step][group] if group in groups[hlt][config][step] else 0)
    
        for s in groups[hlt][config].keys():
            for k in groups[hlt][config][s].keys():
                # rescale and convert units
                groups[hlt][config][s][k] /= Nevents[hlt] * 1000

                # set high-pt triplet step time to 0 for single iter config
                if (s=="HighPtTripletStep") and (config=="PatatrackIT+3OT"):
                    groups[hlt][config][s][k] = 0

In [ ]:
groups

In [ ]:
CATEGORIES = {'IT LocalReco': "Local reconstruction (IT)", 
              'OT LocalReco': "Local reconstruction (OT)", 
              'Pixel Tracking': r"Pixel N-tuplets, $N_{hits}\geq 4$", 
              'Pixel Triplets' : r"Pixel N-tuplets, $N_{hits}= 3$", 
              'Data Conversion': "Data conversion", 
              'High Purity Selection': "High-purity selection"}

## CPU measurements

In [ ]:
hlt=hlts[0]
spec = "Run HLT tracking on\nHLT-filtered set of\nL1-accepted events"
fig, ax = plt.subplots(1, figsize=(10, 10))

STEPS = {"InitialStep": "Initial seeding", "HighPtTripletStep": r"High-$p_\text{T}$ triplet"+"\nseeding"}
bottom = np.zeros(3)
bottomBottom = np.zeros(3)
width = 0.5


for step in steps:
    for category, label in CATEGORIES.items():
        times = np.array([(groups[hlt][c][step][category] if category in groups[hlt][c][step] else 0) for c in configs])
        if step==steps[0]:
            p = ax.bar(configDict.values(), times, width, label=label, bottom=bottom, color = colorDict[category])
        else:
            ax.bar(configDict.values(), times, width, bottom=bottom, color = colorDict[category])
        bottom += times
        if False: #all([category in groups[c][step] for c in configs]):
            ax.plot([width/2, 1-width/2], bottom[:2], color= colorDict[category])
            ax.plot([1+width/2, 2-width/2], bottom[1:], color= colorDict[category])
    scale = 1.5
    #ax.bar(np.arange(len(configDict.values())) - width/2+ width/scale/2, bottom-bottomBottom, width/scale, label=STEPS[step], bottom=bottomBottom, color = colorDict[step])
    alpha = 0.3
    ax.bar([- width/2- width/scale/2], bottom[0]-bottomBottom[0], width/scale, bottom=bottomBottom[0], color = colorDict[step], alpha=alpha)
    ax.text(- width/2- width/scale/2, (bottom[0]+bottomBottom[0])/2, STEPS[step], fontsize=18, horizontalalignment='center', verticalalignment='center', rotation=90)
    ax.fill_between([width/2, 1-width/2], bottom[:2], bottomBottom[:2], color = colorDict[step], alpha=alpha) #, label=STEPS[step]
    if step == steps[0]:
        ax.fill_between([1+width/2, 2-width/2], bottom[1:], bottomBottom[1:], color = colorDict[step], alpha=alpha)
    bottomBottom += bottom

for i, val in enumerate(bottom):
    ax.text(i, val+0.015, (("%.2f" % val) if val>1 else ("%.3f" % val)) + r"$\,\text{s}$", fontsize=18, horizontalalignment='center', verticalalignment='bottom')

ax.set_ylabel("Average computing time\nfor pixel tracking per event [s]")
ax.legend(loc='upper right', bbox_to_anchor=(1-0.02, 0.82), reverse=True)

ax.set_ylim(0, 5.4)
ax.set_xlim(-0.8, 2.5)
#plt.xticks(rotation=-15, ha='left')
ax.set_xticks([], minor=True)

if DRAFT:
    ax.text(0.5, 0.5, 'DRAFT', transform=ax.transAxes,
            fontsize=150, color='gray', alpha=0.25,
            ha='center', va='center', rotation=30)

ax.text(-0.65, 3.7, spec, fontsize=18, horizontalalignment='left', verticalalignment='center')
# add the CMS label
datalabel = r"$\text{t}\bar{\text{t}}$ + 200 PU ($\sqrt{s} = 14\,\text{TeV}$), HLT pixel tracking" + "\nrun on 2x AMD EPYC 9534 (CPU only)" # + "\n" + spec
exptext, expsuffix, supptext, explumi = cmslabel(llabel=THELABEL, rlabel=datalabel, ax=ax, loc=4)
explumi.set_fontsize(explumi.get_fontsize() / 1.25)
for filetype in [".png", ".pdf"]:
    savefig(DIR + hlt + "/bar_total_v2" + filetype)
plt.show()

In [ ]:
# CPU
hlt=hlts[1]
spec = "Run HLT tracking on\nall L1-accepted events" #"filterless HLT:\n- full reconstruction\n  in %s of events" % (r"100$\,$%")
fig, ax = plt.subplots(1, figsize=(10, 10))

STEPS = {"InitialStep": "Initial seeding", "HighPtTripletStep": r"High-$p_\text{T}$ triplet"+"\nseeding"}
bottom = np.zeros(3)
bottomBottom = np.zeros(3)
width = 0.5


for step in steps:
    for category, label in CATEGORIES.items():
        times = np.array([(groups[hlt][c][step][category] if category in groups[hlt][c][step] else 0) for c in configs])
        if step==steps[0]:
            p = ax.bar(configDict.values(), times, width, label=label, bottom=bottom, color = colorDict[category])
        else:
            ax.bar(configDict.values(), times, width, bottom=bottom, color = colorDict[category])
        bottom += times
        if False: #all([category in groups[c][step] for c in configs]):
            ax.plot([width/2, 1-width/2], bottom[:2], color= colorDict[category])
            ax.plot([1+width/2, 2-width/2], bottom[1:], color= colorDict[category])
    scale = 1.5
    #ax.bar(np.arange(len(configDict.values())) - width/2+ width/scale/2, bottom-bottomBottom, width/scale, label=STEPS[step], bottom=bottomBottom, color = colorDict[step])
    alpha = 0.3
    ax.bar([- width/2- width/scale/2], bottom[0]-bottomBottom[0], width/scale, bottom=bottomBottom[0], color = colorDict[step], alpha=alpha)
    ax.text(- width/2- width/scale/2, (bottom[0]+bottomBottom[0])/2, STEPS[step], fontsize=18, horizontalalignment='center', verticalalignment='center', rotation=90)
    ax.fill_between([width/2, 1-width/2], bottom[:2], bottomBottom[:2], color = colorDict[step], alpha=alpha) #, label=STEPS[step]
    if step == steps[0]:
        ax.fill_between([1+width/2, 2-width/2], bottom[1:], bottomBottom[1:], color = colorDict[step], alpha=alpha)
    bottomBottom += bottom

for i, val in enumerate(bottom):
    ax.text(i, val+0.015, (("%.2f" % val) if val>1 else ("%.3f" % val)) + r"$\,\text{s}$", fontsize=18, horizontalalignment='center', verticalalignment='bottom')

ax.set_ylabel("Average computing time\nfor pixel tracking per event [s]")
ax.legend(loc='upper right', bbox_to_anchor=(1-0.02, 0.82), reverse=True)

ax.set_ylim(0, 6.7)
ax.set_xlim(-0.8, 2.5)
#plt.xticks(rotation=-15, ha='left')
ax.set_xticks([], minor=True)

if DRAFT:
    ax.text(0.5, 0.5, 'DRAFT', transform=ax.transAxes,
            fontsize=150, color='gray', alpha=0.25,
            ha='center', va='center', rotation=30)

ax.text(-0.65, 4.3, spec, fontsize=18, horizontalalignment='left', verticalalignment='center')
# add the CMS label
datalabel = r"$\text{t}\bar{\text{t}}$ + 200 PU ($\sqrt{s} = 14\,\text{TeV}$), HLT pixel tracking" + "\nrun on 2x AMD EPYC 9534 (CPU only)" # + "\n" + spec
exptext, expsuffix, supptext, explumi = cmslabel(llabel=THELABEL, rlabel=datalabel, ax=ax, loc=4)
explumi.set_fontsize(explumi.get_fontsize() / 1.25)
for filetype in [".png", ".pdf"]:
    savefig(DIR + hlt + "/bar_total_v2" + filetype)
plt.show()